# CatBoost Baseline Model

This notebook trains a CatBoost classifier using both original dataset features and engineered fraud indicators discovered during exploratory analysis.

The objective is to establish a strong baseline model and identify influential features for subsequent modeling experiments.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/DataSet.csv")

## Feature Engineering

Several binary and interaction features were created based on fraud patterns identified during exploratory analysis.

These features are intended to expose high-risk regions of the feature space that may not be easily learned from raw variables alone.

In [ ]:
df["F115_high"] = (df["F115"] > 0.78).astype(int)

df["F2582_hot"] = (
    (df["F2582"] > -0.03)
    & (df["F2582"] <= 0)
).astype(int)

df["F2956_hot"] = (
    (df["F2956"] > 19)
    & (df["F2956"] <= 32)
).astype(int)

df["F531_hot"] = (
    (df["F531"] > 0.95)
    & (df["F531"] <= 1.35)
).astype(int)

df["F2737_safe"] = (
    (df["F2737"] > 0)
    & (df["F2737"] <= 0.04)
).astype(int)

df["F2582_F531"] = (
    df["F2582_hot"]
    & df["F531_hot"]
).astype(int)

df["fraud_cluster_1"] = (
    df["F2582_hot"]
    & df["F2956_hot"]
    & df["F531_hot"]
).astype(int)

df["f3836_hot"] = (
    (df["F3836"] > 148.596 ) &
    (df["F3836"] <= 20077.212)
).astype(int)

df["F2956_F115"] = (
    df["F2956_hot"] &
    df["F115_high"]
).astype(int)

df["F2582_pos_F2956_low"] = (
    (df["F2582"] > 0.15) &
    (df["F2956"] < 60)
).astype(int)

## Feature Selection

The model uses a combination of original dataset variables and engineered fraud indicators.

Original features capture account behaviour, occupation information, and historical account characteristics, while engineered features represent fraud patterns discovered during exploratory analysis.

In [ ]:
features = [
    "F115",
    "F2582",
    "F2956",
    "F531",
    "F2737",
    "F670",
    "F673",
    "F3891",
    "F3889",

    # engineered features

    "F115_high",
    "F2582_hot",
    "F2956_hot",
    "F531_hot",
    "F2737_safe",
    "F2582_F531",
    "fraud_cluster_1",
    "f3836_hot",
    "F2956_F115",

]

family_cols = [
    #Discovered Feature Families
    
    "F664","F665","F666",
    "F667","F668","F669",
    "F670","F671","F672",
    "F673","F674","F675",

    "F1","F2","F3","F4","F5","F6","F7","F8","F9","F10","F12"
]

for col in family_cols:
    df[f"{col}_missing"] = df[col].isna().astype(int)

features += [f"{col}_missing" for col in family_cols]

In [ ]:
X = pd.get_dummies(
    df[features],
    columns=["F3891", "F3889"],
    dummy_na=True
)

y = df["F3924"]

## Data Preparation

Categorical features are encoded using one-hot encoding and the dataset is divided into training and testing subsets using stratified sampling.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

X_train = X_train.fillna(-999)
X_test = X_test.fillna(-999)

## Model Training

A CatBoost classifier is trained using class weighting to compensate for the severe imbalance between legitimate and fraudulent accounts.

## Evaluation

Model performance is evaluated using precision, recall, F1-score, and confusion matrices. Because fraud detection is a highly imbalanced classification problem, recall on the fraud class is treated as a primary performance metric.

In [ ]:
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, confusion_matrix

fraud_weight = 7200 / 65

cb = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    class_weights=[1, fraud_weight],
    random_state=42,
    verbose=0
)

cb.fit(X_train, y_train)

probs = cb.predict_proba(X_test)[:,1]
preds = (probs >= 0.3).astype(int)

print(confusion_matrix(y_test, preds))
print(classification_report(y_test, preds))

## Feature Importance Analysis

Feature importance scores are examined to understand which variables contribute most strongly to fraud detection.

In [ ]:
cat_importance = pd.Series(
    cb.get_feature_importance(),
    index=X_train.columns
).sort_values(ascending=False)

print(cat_importance.head(15))

## Key Findings

The CatBoost model establishes a strong baseline and highlights several important fraud indicators, including:

- F2737
- F531
- F2956
- F2582
- F115

The results from this model guided the development of subsequent XGBoost, LightGBM, and Neural Network models.